Your task is to create a bert-base-classifier of vacancy areas based on their titles.

Each vacancy can have more than one area so it's **Multi-label classification** not Multiclass classification




In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics import classification_report
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from nltk.tokenize import word_tokenize
from string import punctuation
from tqdm import tqdm

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, RandomSampler, Dataset, SequentialSampler
import random
import transformers

# Try two or more different bert-like models(different berts, robertas etc. or any other transformer based model) (**2 points max**)
 your notebook should contain the training process of all your models!

In [ ]:
MODEL_NAME = 'bert-base-uncased'          # you can swap: 'distilbert-base-uncased', 'albert-base-v2', etc.
MAX_SEQ_LENGTH = 128                      # 512 is BERT hard-limit; 128 balances speed & quality for short job titles
RESULT_MODEL_PATH = './model.pt'

In [ ]:
def seed_everything(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed = 12
seed_everything(seed)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
device

In [ ]:
punctuation = set('!"$%&\'()*,-/:;<=>?@[\\]^_`{|}~') # убрал #

In [ ]:
def clean(text):
    return ' '.join([token.lower() for token in word_tokenize(text) if token not in punctuation])

In [ ]:
df = pd.read_csv('/content/dataset_2020.csv')
df.shape

Each vacancy can have more than one area separated be space

Exapmle:

Malware Analyst for Imunify Security,analyst it_security

In [ ]:
df_train, df_test = train_test_split(df, train_size=0.9, random_state=42)
df_train, df_valid = train_test_split(df_train, train_size=0.8, random_state=42)

# Finish TextClassificationDataset (**1 point max**)

In [ ]:
class TextClassificationDataset(Dataset):
    def __init__(self, data, tokenizer, binarizer):
        self.data = data
        self.tokenizer = tokenizer
        sentences = [clean(sent) for sent in data.title.tolist()]
        self.target = [labels.split() for labels in data.area.tolist()]
        self.binarizer = binarizer
        self.target_one_hot = torch.tensor(self.binarizer.transform(self.target), dtype=torch.float)
        # tokenize once at init (fast)
        self.encodings = self.tokenizer(
            sentences,
            truncation=True,
            padding='max_length',
            max_length=MAX_SEQ_LENGTH,
            return_tensors='pt'
        )

    def __len__(self):
        return len(self.target_one_hot)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.target_one_hot[idx]
        return item

In [ ]:
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
binarizer = MultiLabelBinarizer()
labels_train = [labels.split() for labels in df_train.area.tolist()]
binarizer.fit(labels_train)


In [ ]:
batch_size = 8 # ToDo

train_dataset = TextClassificationDataset(df_train, tokenizer, binarizer)
train_sampler = RandomSampler(train_dataset)
train_dataloader =  DataLoader(train_dataset, sampler=train_sampler, batch_size=batch_size,)

valid_dataset = TextClassificationDataset(df_valid, tokenizer, binarizer)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size)

test_dataset = TextClassificationDataset(df_test, tokenizer, binarizer)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)


In [ ]:
class BertForMultilabel(nn.Module):
    def __init__(self, num_labels: int):
        super().__init__()
        self.bert = transformers.BertModel.from_pretrained(MODEL_NAME)

        # TODO: add your custom layers here
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def train_bert(self, train_bert_flag=True):
        for param in self.bert.parameters():
            param.requires_grad = train_bert_flag

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None):
        # TODO: implement forward pass
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        pooled = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        return logits

In [ ]:
num_labels = len(binarizer.classes_)
model = BertForMultilabel(num_labels)
model.to(device)
;

# Train your classifier with freezed bert and save model with the lowest val loss during training (**2 points max**)

print train/val loss after each epoch


In [ ]:
def train(model, iterator, optimizer, criterion):
    model.train()
    epoch_loss = 0.0

    for batch in iterator:
        # move tensors to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # zero grads
        optimizer.zero_grad()

        # forward
        logits = model(input_ids=input_ids, attention_mask=attention_mask)

        # compute loss
        loss = criterion(logits, labels)

        # backward
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(iterator)

In [ ]:
def validate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0.0

    with torch.no_grad():
        for batch in iterator:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)

            epoch_loss += loss.item()

    return epoch_loss / len(iterator)

In [ ]:
def logits_to_labels(logits):
    preds = nn.Sigmoid()(logits.view(-1, num_labels))
    preds = preds.to('cpu').numpy()>0.5
    return preds.tolist()

In [ ]:
model.train_bert(False)

In [ ]:
epochs = 3 # ToDo
criterion = nn.BCEWithLogitsLoss()# ToDo what criterion do you need for multilabel classification?
optimizer  = torch.optim.Adam(model.parameters(), lr=2e-5) # ToDo use adam optimizer
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.1) # ToDo use StepLR scheduler

In [ ]:
# ToDo Train your model
# ---------- training with early-stopping / best-val-loss save ----------
best_val_loss = float('inf')

for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")

    # training pass
    train_loss = train(model, train_dataloader, optimizer, criterion)
    print(f"  train loss: {train_loss:.4f}")

    # validation pass
    val_loss = validate(model, valid_dataloader, criterion)
    print(f"  val   loss: {val_loss:.4f}")

    # Step the scheduler
    scheduler.step()

    # save checkpoint if validation improved
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), RESULT_MODEL_PATH)
        print("  🎯  New best val loss -> model saved")

print("Training finished. Lowest val loss:", best_val_loss)

In [ ]:
model.load_state_dict(torch.load(RESULT_MODEL_PATH, map_location=torch.device(device)))
model.eval()  # inference mode

# run the test set through the model to get logits
test_logits = []
with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        test_logits.append(logits)

test_logits = torch.cat(test_logits, dim=0)  # shape: (N_samples, num_labels)

# convert logits → binary predictions
test_preds = logits_to_labels(test_logits)
print("Test predictions shape:", len(test_preds), "samples")

In [ ]:
print(classification_report(
    binarizer.transform(test_dataset.target),  # ground-truth multi-hot
    test_preds,                                # predicted multi-hot
    target_names=binarizer.classes_))

# Train your classifier with unfreezed bert and save model with the lowest val loss during training (**2 points max**)

print train/val loss after each epoch

In [ ]:
# unfreeze BERT → full fine-tuning
model.train_bert(train_bert_flag=True)

epochs = 4          # 3-5 is usually enough for full fine-tuning
lr = 2e-5           # standard BERT fine-tune LR
WARMUP_PROPORTION = 0.1
warmup_steps = int(len(train_dataloader) * epochs * WARMUP_PROPORTION)
print(f"Warm-up steps: {warmup_steps}")

In [ ]:
model.train_bert(True)

In [ ]:
# no decay on bias and LayerNorm weight (standard BERT trick)
no_decay = ['bias', 'LayerNorm.weight']

param_optimizer = list(model.named_parameters())
optimizer_grouped_parameters = [
    {'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)],
     'weight_decay': 0.001},
    {'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)],
     'weight_decay': 0.0}
]

criterion = nn.BCEWithLogitsLoss()       # multi-label loss
lr = 2e-5                                # learning rate

# FIX: Use torch.optim.AdamW instead of transformers.optimization.AdamW
optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=lr)

# Calculate total training steps
t_total = len(train_dataloader) * epochs

scheduler = transformers.optimization.get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=t_total
)

In [ ]:
model.load_state_dict(torch.load(RESULT_MODEL_PATH, map_location=torch.device(device)))
test_preds = validate(model, test_dataloader, criterion)

In [ ]:
print(classification_report(binarizer.transform(test_dataset.target), test_preds,
                            target_names=binarizer.classes_))

In [ ]:
# Results

# Results (3 points max)

Write your conclusion

What models and what training parameters did you use?

What was the reason for your choice?

What were the results?

What metrics do you consider the most important?